# Flow Matching on $\mathbb{R}^2$
In this notebook, we perform flow matching on the two dimensional translation group $\mathbb{R}^2$, which is isometric to two dimensional Euclidean space.

In [ ]:
import torch
from torch.optim.swa_utils import AveragedModel, get_ema_multi_avg_fn
import numpy as np
from lieflow.groups import Rn
from lieflow.models import (
    get_model_FM,
    LogarithmicDistance,
    DataGenerator,
)
from lieflow.metrics import compare
import matplotlib.pyplot as plt
from matplotlib.animation import FFMpegWriter
import matplotlib.colors as mcolors
import matplotlib.gridspec as gridspec
from matplotlib.cm import ScalarMappable
from tqdm.notebook import tqdm
from functools import partial
import json
with open("plot.json", "r") as f:
    plot_config = json.load(f)
latex_line_width = plot_config.pop("latex_line_width")
plt.rcParams.update(plot_config)

device = "cpu"
SEED = 0
torch.manual_seed(SEED);

In [ ]:
r2 = Rn(2)

In [ ]:
generate_videos = True

# Distributions

In [ ]:
def generate_normals(N):
    return torch.randn(N, 2, device=device)

def generate_uniforms_on_circle(N, ε=0.04):
    normals = generate_normals(N)
    lengths = (normals**2).sum(dim=-1).sqrt()
    non_zero = lengths > 0.
    circle_samples = normals[non_zero] / lengths[non_zero, None]
    return circle_samples + ε * torch.randn(circle_samples.shape, device=device)

def plot_x(ax, x, **kwargs):
    x = x.to("cpu")
    return ax.scatter(x[..., 0], x[..., 1], **kwargs)

In [ ]:
tests = ("normals_to_circle",)

In [ ]:
def data_generator(test, ε=0.04):
    match test:
        case "normals_to_circle":
            generate_x_0 = generate_normals
            generate_x_1 = partial(generate_uniforms_on_circle, ε=ε)
    return generate_x_0, generate_x_1

# Models

## Training

In [ ]:
with open("config.json", "r") as f:
    config_all = json.load(f)
config_space = config_all["R2"]
# Default parameters, overridden by those of the space, which are in turn overridden by those of the test case.
config_shared = config_all["defaults"] | {key: value for key, value in config_space.items() if not isinstance(value, dict)}
config = {test: config_shared | value for test, value in config_space.items() if isinstance(value, dict)}

In [ ]:
N_EVAL = 2**8

eval_data = {}
for test in tests:
    generate_x_0, generate_x_1 = data_generator(test, ε=config[test]["epsilon"])
    x_0s_eval, x_1s_eval, x_1s_ref = generate_x_0(N_EVAL), generate_x_1(N_EVAL), generate_x_1(N_EVAL)
    eval_data[test] = (x_0s_eval, x_1s_eval, compare(r2, x_1s_ref, x_1s_eval))

In [ ]:
def train_model(generate_x_0, generate_x_1, config, x_0s_eval, x_1s_eval, floor):
    train_loader = DataGenerator(generate_x_0, generate_x_1, config["batch_size"], config["batches_per_epoch"])

    model_FM = get_model_FM(
        r2, H=config["width"], L=config["depth"], shared_weights=config["shared_weights"],
        activation=getattr(torch.nn, config["activation"])
    ).to(device)
    print("Number of parameters: ", model_FM.parameter_count)
    
    optimizer_FM = torch.optim.Adam(model_FM.parameters(), config["learning_rate"], weight_decay=config["weight_decay"])
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer_FM, T_max=config["epochs"] * config["batches_per_epoch"], eta_min=config["learning_rate_final"]
    )
    ema = AveragedModel(model_FM, multi_avg_fn=get_ema_multi_avg_fn(config["ema_decay"]))
    loss = LogarithmicDistance(torch.tensor(config["loss_weights"], device=device))

    losses_FM = np.zeros(config["epochs"])
    metrics_FM = {key: np.zeros(config["epochs"]) for key in floor}
    for i in tqdm(range(config["epochs"])):
        losses_FM[i] = model_FM.train_network(device, train_loader, optimizer_FM, scheduler, ema, loss, show_progress=False)
        for key, value in ema.module.evaluate(x_0s_eval, x_1s_eval, N_steps=config["N_steps"]).items():
            metrics_FM[key][i] = value

    epochs = np.arange(1, config["epochs"] + 1)
    fig, ax = plt.subplots(1, 3, figsize=(15, 4.5))
    ax[0].plot(epochs, losses_FM)
    ax[0].set_title("Batch Loss Flow Matching")
    ax[0].set_yscale("log")
    labels = {"W_1": "$W_1$", "precision": "precision", "coverage": "coverage"}
    for (key, label), color in zip(labels.items(), ("tab:blue", "tab:orange", "tab:green")):
        ax[1].plot(epochs, metrics_FM[key], color=color, label=label)
        ax[1].axhline(floor[key], color=color, linestyle="--", linewidth=0.8)
    ax[1].set_title("Distances (dashed: floor)")
    ax[1].set_yscale("log")
    ax[1].legend()
    ax[2].plot(epochs, metrics_FM["MMD^2"], color="tab:red")
    ax[2].axhline(floor["MMD^2"], color="tab:red", linestyle="--", linewidth=0.8)
    ax[2].set_title("MMD$^2$ (dashed: floor)")
    for a in ax:
        a.set_xscale("log")
        a.set_xlabel("Epoch")

    return ema.module, losses_FM, metrics_FM

In [ ]:
models_FM = {}
losses_FM = {}
metrics_FM = {}
for test in tests:
    print(test)
    generate_x_0, generate_x_1 = data_generator(test, ε=config[test]["epsilon"])

    x_0s = generate_x_0(32)
    x_1s = generate_x_1(32)

    fig, ax = plt.subplots(1, 2, figsize=(10, 5))
    plot_x(ax[0], x_0s[:32])
    ax[0].set_xlim(-3, 3)
    ax[0].set_ylim(-3, 3)
    ax[0].set_title(r"$\mathfrak{X}_0$")
    plot_x(ax[1], x_1s[:32])
    ax[1].set_xlim(-3, 3)
    ax[1].set_ylim(-3, 3)
    ax[1].set_title(r"$\mathfrak{X}_1$")

    models_FM[test], losses_FM[test], metrics_FM[test] = train_model(
        generate_x_0, generate_x_1, config[test], *eval_data[test]
    )

# Testing

In [ ]:
def create_animations(tests, models_FM, N_samples=2**5, N_steps=120):
    N_models = len(tests)

    x_0s_test = {}
    x_ts_test = {}
    for test in tests:
        generate_x_0, _ = data_generator(test, ε=config[test]["epsilon"])
        x_0s_test[test] = generate_x_0(N_samples).to(device)
        x_ts_test[test] = x_0s_test[test].detach().clone()
        
        models_FM[test].eval()

    t = 0
    Δt = 1. / N_steps

    for test in tests:
        metadata = {'title': 'Flow Matching R^2', 'artist': 'Matplotlib'}
        writer = FFMpegWriter(fps=30, metadata=metadata)

        fig, ax = plt.subplots(1, N_models, figsize=(5 * N_models, 5))
        ax.set_xlim(-3, 3)
        ax.set_ylim(-3, 3)

        with writer.saving(fig, f"output/flow_matching_R2_{test}.mp4", dpi=150):
            scatter = ax.scatter([], [])
            for frame in tqdm(range(N_steps + 1)):
                scatter.remove()
                scatter = plot_x(ax, x_ts_test[test], color="tab:blue")
                ax.set_title(fr"$\mathfrak{{X}}_{{{t:.2f}}}$")
                writer.grab_frame()
                t = frame * Δt
                with torch.no_grad():
                    x_ts_test[test] = models_FM[test].step(x_ts_test[test], torch.tensor([t], device=device), Δt)

In [ ]:
if generate_videos:
    create_animations(tests, models_FM, N_samples=2**5)

In [ ]:
t = 0
N_steps = config_shared["N_steps"]
Δt = 1. / N_steps
N_show = 5
N_skip = int(N_steps / (N_show-1))

N_models = len(tests)
N_samples = 32

x_0s_test = {}
x_1s_test = {}
x_ts_test = {}
for test in tests:
    generate_x_0, generate_x_1 = data_generator(test, ε=config[test]["epsilon"])
    x_0s_test[test] = generate_x_0(N_samples).to(device)
    x_1s_test[test] = generate_x_1(N_samples).to(device)
    x_ts_test[test] = x_0s_test[test].detach().clone()
    
    models_FM[test].eval()

fig = plt.figure(figsize=(4.8, 1.6 * N_models * 3/3.1))
gs = gridspec.GridSpec(N_models, 4, width_ratios=[1, 1, 1, 0.1], height_ratios=N_models * [1.], wspace=0.1, hspace=0.1)
cax = fig.add_subplot(gs[:, 3])

ax = []
for i in range(N_models):
    ax.append([])
    for j in range(3):
        a = fig.add_subplot(gs[i, j])
        a.set_xlim(-3, 3)
        a.set_ylim(-3, 3)
        a.set_xticks([])
        a.set_yticks([])
        a.set_aspect("equal")
        ax[i].append(a)
        
ax[0][0].set_title(r"$\mathfrak{X}_0$")
ax[0][1].set_title(r"$\mathfrak{X}_t$")
ax[0][2].set_title(r"$\mathfrak{X}_1$")
Δc = 1 / (N_show - 1)
colors = [(j * Δc, 0.1, 1 - j * Δc) for j in range(N_show)]
cmap = mcolors.ListedColormap(colors)
for i, test in enumerate(tests):
    k = 0

    plot_x(ax[i][0], x_0s_test[test][:N_samples], marker=".")
    plot_x(ax[i][2], x_1s_test[test][:N_samples], marker=".")

    alpha = 1
    N_samples_shown = N_samples
    for j in range(N_steps+1):
        t = j * Δt
        if j == N_steps:
            alpha = 1
            N_samples_shown = N_samples
        with torch.no_grad():
            x_ts_test[test] = models_FM[test].step(x_ts_test[test], torch.tensor([t], device=device), torch.tensor([Δt], device=device))

        if j % N_skip == 0:
            im = plot_x(ax[i][1], x_ts_test[test], color=colors[k], marker=".", alpha=alpha)
            k += 1
        alpha = 0.5
        N_samples_shown = N_samples // 4
fig.colorbar(ScalarMappable(cmap=cmap), cax=cax, ticks=np.linspace(0, 1, N_show), label="$t$");
fig.savefig("output/interpolation_R2.pdf", bbox_inches="tight")

In [ ]:
N_EVAL_FINAL = 2**10

def print_metrics(name, metrics):
    print(f"{name:>10}: " + "  ".join(f"{key} = {value:.4f}" for key, value in metrics.items()))

eval_data_final = {}
for test in tests:
    generate_x_0, generate_x_1 = data_generator(test, ε=config[test]["epsilon"])
    x_0s_eval, x_1s_eval, x_1s_ref = generate_x_0(N_EVAL_FINAL), generate_x_1(N_EVAL_FINAL), generate_x_1(N_EVAL_FINAL)
    floor = compare(r2, x_1s_ref, x_1s_eval)
    eval_data_final[test] = (x_0s_eval, x_1s_eval, floor)

    print(test)
    print_metrics("floor", floor)
    print_metrics("model", models_FM[test].evaluate(x_0s_eval, x_1s_eval, N_steps=config[test]["N_steps"]))
    print_metrics("no model", compare(r2, x_0s_eval, x_1s_eval))

In [ ]:
steps = 2**np.arange(9)
keys = {"W_1": "$W_1$", "precision": "precision", "coverage": "coverage"}

fig, ax = plt.subplots(
    1, len(tests), figsize=(latex_line_width, 0.45 * latex_line_width), sharey=True, squeeze=False
)
ax = ax[0]
for a, test in zip(ax, tests):
    x_0s_eval, x_1s_eval, floor = eval_data_final[test]
    metrics = [models_FM[test].evaluate(x_0s_eval, x_1s_eval, N_steps=n) for n in tqdm(steps)]
    for (key, label), color in zip(keys.items(), ("tab:blue", "tab:orange", "tab:green")):
        a.plot(steps, [m[key] for m in metrics], marker=".", color=color, label=label)
        a.axhline(floor[key], color=color, linestyle="--", linewidth=0.8)
    a.set_xscale("log", base=2)
    a.set_yscale("log")
    a.set_xlabel("Integration steps")
    a.set_title(test.replace("_", " "))
ax[0].set_ylabel("Distance")
ax[0].legend();